In [1]:
import requests
import pandas as pd
from getpass import getpass

pd.set_option("display.max_columns", None)

AUTH_KEY = getpass("KRX AUTH_KEY 입력: ")

BASE_URL = (
    "https://data-dbg.krx.co.kr/"
    "svc/apis/sto/ksq_bydd_trd"
)

TEST_DATE = "20261012"

KRX AUTH_KEY 입력:  ········


In [2]:
def get_kosdaq_daily_trade(auth_key, base_date):

    headers = {
        "AUTH_KEY": auth_key
    }

    params = {
        "basDd": base_date
    }

    try:
        response = requests.get(
            BASE_URL,
            headers=headers,
            params=params,
            timeout=60
        )

        print("HTTP 상태코드:", response.status_code)

        if response.status_code != 200:
            print("API 호출 실패")
            print(response.text[:2000])
            return pd.DataFrame()

        data = response.json()

        rows = data.get("OutBlock_1", [])

        if not rows:
            print("조회 결과가 없습니다.")
            return pd.DataFrame()

        return pd.DataFrame(rows)

    except requests.exceptions.ReadTimeout:
        print("서버 응답시간 초과")
        return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print("API 요청 오류:", e)
        return pd.DataFrame()

In [3]:
df = get_kosdaq_daily_trade(
    auth_key=AUTH_KEY,
    base_date=TEST_DATE
)

print("조회 건수:", len(df))

if not df.empty:
    print(df.columns.tolist())
    display(df.head())

HTTP 상태코드: 200
조회 건수: 1821
['BAS_DD', 'ISU_CD', 'ISU_NM', 'MKT_NM', 'SECT_TP_NM', 'TDD_CLSPRC', 'CMPPREVDD_PRC', 'FLUC_RT', 'TDD_OPNPRC', 'TDD_HGPRC', 'TDD_LWPRC', 'ACC_TRDVOL', 'ACC_TRDVAL', 'MKTCAP', 'LIST_SHRS']


,BAS_DD,ISU_CD,ISU_NM,MKT_NM,SECT_TP_NM,TDD_CLSPRC,CMPPREVDD_PRC,FLUC_RT,TDD_OPNPRC,TDD_HGPRC,TDD_LWPRC,ACC_TRDVOL,ACC_TRDVAL,MKTCAP,LIST_SHRS
0,20261012,060310,3S,KOSDAQ,벤처기업부,1253,6,0.48,1247,1280,1200,205053,258129182,66482977120,53059040
1,20261012,054620,APS,KOSDAQ,중견기업부,3865,-115,-2.89,4075,4075,3810,12973,50342480,71093664165,18394221
2,20261012,079810,APS이노베이션,KOSDAQ,우량기업부,3560,-120,-3.26,3695,3800,3530,69226,249144542,76640014640,21528094
3,20261012,265520,AP시스템,KOSDAQ,우량기업부,19450,-1150,-5.58,20550,21000,19410,127297,2518372440,292738468450,15050821
4,20261012,211270,AP위성,KOSDAQ,우량기업부,7600,-420,-5.24,8020,8200,7570,117181,908554660,114625510400,15082304


In [4]:
target = df[
    df["ISU_NM"].astype(str).str.strip() == "원익IPS"
].copy()

display(target)

,BAS_DD,ISU_CD,ISU_NM,MKT_NM,SECT_TP_NM,TDD_CLSPRC,CMPPREVDD_PRC,FLUC_RT,TDD_OPNPRC,TDD_HGPRC,TDD_LWPRC,ACC_TRDVOL,ACC_TRDVAL,MKTCAP,LIST_SHRS
1152,20261012,240810,원익IPS,KOSDAQ,우량기업부,130900,-2800,-2.09,130900,138600,127100,1528053,200805837880,6425082640900,49083901


In [5]:
display(
    target[
        [
            "BAS_DD",
            "ISU_CD",
            "ISU_NM",
            "MKT_NM",
            "TDD_OPNPRC",
            "TDD_HGPRC",
            "TDD_LWPRC",
            "TDD_CLSPRC",
            "ACC_TRDVOL",
            "ACC_TRDVAL",
            "MKTCAP"
        ]
    ]
)

,BAS_DD,ISU_CD,ISU_NM,MKT_NM,TDD_OPNPRC,TDD_HGPRC,TDD_LWPRC,TDD_CLSPRC,ACC_TRDVOL,ACC_TRDVAL,MKTCAP
1152,20261012,240810,원익IPS,KOSDAQ,130900,138600,127100,130900,1528053,200805837880,6425082640900


In [6]:
print("===== D12 검증 결과 =====")

print("API 데이터 존재:", not df.empty)
print("코스닥시장 조회 건수:", len(df))
print("원익IPS 검색 결과:", len(target))

if not target.empty:
    print("조회 기준일:", target["BAS_DD"].iloc[0])
    print("원익IPS 종가:", target["TDD_CLSPRC"].iloc[0])
    print("원익IPS 거래량:", target["ACC_TRDVOL"].iloc[0])

===== D12 검증 결과 =====
API 데이터 존재: True
코스닥시장 조회 건수: 1821
원익IPS 검색 결과: 1
조회 기준일: 20261012
원익IPS 종가: 130900
원익IPS 거래량: 1528053


## D12 검증 결과

| 검증 항목 | 결과 |
|---|---|
| KRX API 연결 | 성공 |
| 코스닥시장 일별 데이터 조회 | 성공 |
| 코스닥 종목 추출 | 성공 |
| 시가·고가·저가·종가 | 확인 |
| 거래량·거래대금 | 확인 |
| 장기 과거 데이터 접근 | 추가 확인 가능 |

### 주의사항
- 코스닥 일별매매정보 API는 별도 활용승인이 필요하다.
- 인증키는 요청 Header의 `AUTH_KEY`로 전달한다.
- 2010-01-04 이후 데이터를 제공한다.